In [1]:
from typing import NamedTuple

import numpy as np
import numpy.typing as npt
import pytest

from clarautils.QueryableTable import (
    QueryableTable,
    ConstraintColumn,
    Constraint,
    Query,
    Undefined,
)

from clarautils import CCol, TableFields

Define Table Datatype

In [2]:
class DTableFields(TableFields):
    signed: CCol | np.bool_
    abs_min: CCol | np.uint64
    max: CCol | np.uint64
    bits: CCol | np.uint8
    type: CCol | np.object_

Build data to hold

In [3]:
def build_type_tbl():
    kind = {'u': False, 'i': True}
    sizes = [1, 2, 4, 8]
    types = np.array([
        np.dtype(f"{k}{s}")
        for k in kind
        for s in sizes
    ])

    return [(
        kind[t.kind],
        -np.iinfo(t).min,
        np.iinfo(t).max,
        np.iinfo(t).bits,
        type
    ) for t in types]

data = build_type_tbl()
print(data)

[(False, 0, 255, 8, <class 'type'>), (False, 0, 65535, 16, <class 'type'>), (False, 0, 4294967295, 32, <class 'type'>), (False, 0, 18446744073709551615, 64, <class 'type'>), (True, 128, 127, 8, <class 'type'>), (True, 32768, 32767, 16, <class 'type'>), (True, 2147483648, 2147483647, 32, <class 'type'>), (True, 9223372036854775808, 9223372036854775807, 64, <class 'type'>)]


stick it together

In [7]:
qtbl = QueryableTable(data, DTableFields)
print(qtbl)

QueryableTable(len=8)


The table is typed, so accessing the columns is as easy as it should be, with full pycharm linting support

In [11]:
print(qtbl.signed)

CCol(signed): array([False, False, False, False,  True,  True,  True,  True])


Acces a row

In [9]:
row = qtbl[0]
print(row)

DTable(signed=np.False_, abs_min=np.uint64(0), max=np.uint64(255), bits=np.uint8(8), type=<class 'type'>)


Row is typed so working with it without nasty workarounds

In [12]:
if row.signed:
    print("Is signed")
else:
    print("Not signed")

Not signed


Actual querying

In [19]:
item = qtbl.type.get_first((qtbl.signed == True) & (qtbl.max > np.iinfo(np.int16).max))
print(np.dtype(item))

object
